In [127]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [3]:
df = pd.read_csv("./data/Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [13]:
X = df.drop(columns=["Exited"])
y = df["Exited"]

In [14]:
# dropping the some columns which is n't necessary 
X.drop(["RowNumber","CustomerId","Surname"],axis=1,inplace=True)

In [15]:
X.columns 

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'],
      dtype='str')

In [18]:
X["Gender"] = X["Gender"].map({"Male":1,"Female":0})

In [21]:
ohe = OneHotEncoder()
geography_table  = ohe.fit_transform(X[["Geography"]])

In [25]:
geography_df = pd.DataFrame(data=geography_table.toarray(),columns=ohe.get_feature_names_out(),dtype="int")

In [27]:
# concating with X
X = pd.concat([X,geography_df],axis=1)
X.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,France,0,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,France,0,39,1,0.00,2,0,0,93826.63,1,0,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0,1


In [28]:
# dropping the geography columns
X.drop(["Geography"],axis=1,inplace=True)

In [30]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [31]:
# scaling the data
scalar = StandardScaler()
X_train_scale = scalar.fit_transform(X_train)
X_test_scale = scalar.transform(X_test)

#### ANN implementation


In [38]:
X.shape 

(10000, 12)

In [34]:
# sequential model
# initilize the squential network
# initilze the hidden neuron with dense
# activation function (Relu and sigmoid)
# optimizer(Adam) for backward propagation
# loss function (classification: log loss)
# metrix
# traning - logs - display the logs with tensorboard

In [80]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping ,TensorBoard
import datetime 

In [99]:
# total input columns 
input_size = X_train_scale.shape[1]
input_size

12

In [100]:

""" 
    input size will the total number of the columns 
    first hidden layers will have 64 layers 
    and second hidden layers will have 32 hidden layers
    output layers will have only 1 layers
    for hidden layers -> Relu is use as a activation function
    Output is binary so sigmoid activation function using 
    """

# build ANN model
model = Sequential(
    [
        # find layers
        Dense(64,activation="relu",input_shape=(X_train_scale.shape[1],)),
        # hidden layers
        Dense(32,activation="relu"),
        # output layers
        Dense(1,activation="sigmoid")
    ]
)

/Users/bhaskarrai/Desktop/ann_tesnsor/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [102]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [103]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()
# definig both optimizer and loss function 

In [104]:
# for forward and backward we need to comple the model
model.compile(optimizer=optimizer,loss="binary_crossentropy",metrics=["accuracy"])

In [74]:
import datetime
import os 

In [82]:
file_name = f"{datetime.datetime.now().strftime("%Y_%m_%d_%M%H%S")}.log"
file_path = os.path.join("logs",file_name)

In [105]:
tensorflow_callback = TensorBoard(log_dir=os.path.dirname(file_path),histogram_freq=1)

In [106]:
# setup early stoping 
# while traning the model we see the losss value 
# after some points loss value will stop decreasing if loss value doesn't decrease after the some epoches then stop it
early_stopping_callback = EarlyStopping(monitor="val_loss",patience=15,restore_best_weights=True)

In [107]:
ann = model.fit(
    X_train_scale,y_train,
    validation_data=(X_test_scale,y_test),
    epochs=100,
    callbacks = [tensorflow_callback,early_stopping_callback]
    )

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8349 - loss: 0.4009 - val_accuracy: 0.8530 - val_loss: 0.3478
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 989us/step - accuracy: 0.8541 - loss: 0.3532 - val_accuracy: 0.8525 - val_loss: 0.3552
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 797us/step - accuracy: 0.8587 - loss: 0.3457 - val_accuracy: 0.8585 - val_loss: 0.3449
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step - accuracy: 0.8591 - loss: 0.3430 - val_accuracy: 0.8525 - val_loss: 0.3451
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step - accuracy: 0.8610 - loss: 0.3378 - val_accuracy: 0.8530 - val_loss: 0.3506
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 770us/step - accuracy: 0.8618 - loss: 0.3371 - val_accuracy: 0.8510 - val_loss: 0.3481
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 841us/step - accuracy: 0.8620 - loss: 0.3362 - val_accuracy: 0.8595 - val_loss: 0.3422
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 779us/step - accuracy: 0.8625 - loss: 0.3

In [109]:
# saving the mode
model_dir = "model"
model_name = "a.h5"
model_path = os.path.join(model_dir,model_name)

In [111]:
model.save(model_path)

In [125]:
%reload_ext tensorboard 

In [123]:
%tensorboard --logdir logs/train 

Reusing TensorBoard on port 6007 (pid 60425), started 0:00:24 ago. (Use '!kill 60425' to kill it.)

In [126]:
%tensorboard --logdir logs/validation

Reusing TensorBoard on port 6008 (pid 60486), started 0:00:26 ago. (Use '!kill 60486' to kill it.)

In [128]:
# saving pickel file.
def save_pickle(obj,file_path):
    try:
        dir_path = os.path.dirname(file_path)
        os.makedirs(dir_path,exist_ok=True)
        with open(file_path,"wb") as file:
            pickle.dump(obj,file)
    except Exception as e:
        print(e)

In [129]:
save_pickle(scalar,"model/pickle/scalar.pkl")
save_pickle(ohe,"model/pickle/ohe.pkl")

In [138]:
prediction_data = df.iloc[:1,:]

In [139]:
prediction_data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1


In [143]:
geo_obe_df = ohe.transform([prediction_data["Geography"]])

/Users/bhaskarrai/Desktop/ann_tesnsor/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [150]:
new_geo_df = pd.DataFrame(geo_obe_df.toarray(),columns=ohe.get_feature_names_out(),dtype="int")

In [151]:
new_geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1,0,0


In [152]:
p = pd.concat([prediction_data,new_geo_df],axis=1)

In [153]:
p

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,101348.88,1,1,0,0


In [155]:
# dropping the some columns which isn't require 
p.drop(["RowNumber","CustomerId","Surname","Geography"],axis=1,inplace=True)

In [158]:
p["Gender"]=0

In [160]:
p.shape 

(1, 13)

In [162]:
X_p = p.drop(["Exited"],axis=1)
y = p["Exited"]

In [164]:
p_scale = scalar.transform(X_p)

In [166]:
p_scale

array([[-0.33880827, -1.09499335,  0.29493847, -1.04241787, -1.21847056,
        -0.91668767,  0.64920267,  0.97481699,  0.01595384,  1.00150113,
        -0.57946723, -0.57638802]])

In [167]:
prediction = model.predict(p_scale)
prediction 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


array([[0.42180523]], dtype=float32)

In [168]:
y 

0    1
Name: Exited, dtype: int64

In [171]:
prediction_prop = prediction[0][0]
error = y-prediction_prop
print(error)

0    0.578195
Name: Exited, dtype: float64
